# Learn Mctx by building a batched tic-tac-toe agent

This tutorial adapts the tic-tac-toe rules from the parent MCTS project to DeepMind's JAX-native Mctx interface. We solve tactical positions, play complete games, and benchmark batched search on the JAX backend available inside the Mctx Docker image.

- Reusable JAX and Mctx code lives in `mctx_tic_tac_toe_utils.py`
- The existing `alphazero_utils.py` remains the game and rollout-MCTS baseline
- Uniform policy priors and zero leaf values keep this milestone focused on search rather than neural-network training

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import random

import jax
import jax.numpy as jnp
import numpy as np

import helpers.hdbg as hdbg
import helpers.hnotebook as hnotebook

_LOG = logging.getLogger(__name__)

hdbg.init_logger(verbosity=logging.INFO)
hnotebook.config_notebook()

In [ ]:
import research.Implement_MonteCarlo_Tree_Search_and_Alpha_Zero.alphazero_utils as rimtsaazau
import research.Implement_MonteCarlo_Tree_Search_and_Alpha_Zero.mctx.mctx_tic_tac_toe_utils as rimtsaazmmtttu

# Part 1: Confirm the accelerator

Mctx uses the same Python API on CPU and GPU. The Docker launch scripts select an NVIDIA GPU automatically when one is available and otherwise set JAX to its CPU backend. Always inspect the backend before interpreting a timing result.

In [ ]:
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())

# Part 2: Translate tic-tac-toe into the Mctx model interface

Mctx does not ask for mutable Python tree nodes. It asks for a batched root and a recurrent function that predicts what happens after an action. Boards are canonicalized so `1` always represents the player about to move. Legal actions receive uniform logits and occupied cells are masked.

## Cell 2.1: Inspect a root batch

In [ ]:
game = rimtsaazau.TicTacToe()
demo_state = (1, 1, 0, -1, -1, 0, 0, 0, 0)
root, invalid_actions = rimtsaazmmtttu.build_mctx_root([demo_state])

print(game.render(demo_state))
print("\nCanonical embedding:", np.asarray(root.embedding)[0])
print("Invalid actions:    ", np.asarray(invalid_actions)[0])
print("Prior logits:       ", np.asarray(root.prior_logits)[0])

## Cell 2.2: Apply the winning move through the recurrent model

X wins immediately by playing cell 2. Our adapter returns reward `1` and discount `0` for that transition. It encodes the player switch on nonterminal moves with discount `-1`; Mctx does not infer this game rule. Terminal boards use finite dummy logits and absorbing transitions: further search expansion leaves the board unchanged with zero reward and discount. These placeholders are not playable moves.

In [ ]:
recurrent_output, next_embedding = rimtsaazmmtttu.mctx_recurrent_fn(
    (),
    jax.random.PRNGKey(0),
    jnp.asarray([2]),
    root.embedding,
)
print("Reward:        ", np.asarray(recurrent_output.reward))
print("Discount:      ", np.asarray(recurrent_output.discount))
print("Next embedding:", np.asarray(next_embedding)[0])

# Part 3: Search and inspect one tactical position

## Cell 3.1: Mctx finds the immediate win

`run_mctx_search()` uses deterministic Gumbel MuZero search for this perfect-information evaluation. The returned policy output includes the chosen action, normalized action weights, and the complete search tree.

In [ ]:
policy_output = rimtsaazmmtttu.run_mctx_search(
    [demo_state], num_simulations=128, seed=0
)
summary = policy_output.search_tree.summary()
selected_action = int(np.asarray(policy_output.action)[0])

print("Selected action:", selected_action)
print("Action weights: ", np.asarray(policy_output.action_weights)[0])
print("Visit counts:   ", np.asarray(summary.visit_counts)[0])
print("Q-values:       ", np.asarray(summary.qvalues)[0])

**Key observations**:

- The selected action is cell 2, which completes X's top row
- Illegal actions retain their fixed tensor positions but receive no search probability
- The action weights can later become training targets for an AlphaZero policy network

# Part 4: Search several positions together

Batching is the central difference between this implementation and the Python rollout tree. The positions below include both X-to-move and O-to-move states, but canonicalization lets one recurrent model search them together.

In [ ]:
batch_states = [
    demo_state,
    (1, -1, 1, 1, -1, 0, 0, 0, 0),
    (1, 1, 0, 0, -1, 0, 0, 0, 0),
]
batch_output = rimtsaazmmtttu.run_mctx_search(
    batch_states, num_simulations=128, seed=1
)
for state, action in zip(batch_states, np.asarray(batch_output.action)):
    print(game.render(state))
    print("Mctx action:", int(action))
    print()

# Part 5: Play complete games

The Mctx player follows the same `(game, state) -> move` contract as the parent project's random and rollout-MCTS players. We can therefore reuse `play_game()` and `evaluate_win_rate()` unchanged.

## Cell 5.1: Mctx vs. random

In [ ]:
mctx_player = rimtsaazmmtttu.make_mctx_player(
    num_simulations=128, seed=0
)
winner, _ = rimtsaazau.play_game(
    game, mctx_player, rimtsaazau.random_player, verbose=True
)
print("\nWinner:", winner)

## Cell 5.2: Compare outcome rates

Both players receive 128 simulations per move. The number is easy to compare, but each simulation does different work: the original implementation performs a random rollout, whereas Mctx expands its model-based JAX tree.

In [ ]:
random.seed(0)
mctx_results = rimtsaazau.evaluate_win_rate(
    game,
    rimtsaazmmtttu.make_mctx_player(num_simulations=128, seed=0),
    rimtsaazau.random_player,
    num_games=30,
)
random.seed(0)
rollout_results = rimtsaazau.evaluate_win_rate(
    game,
    rimtsaazau.make_mcts_player(num_simulations=128),
    rimtsaazau.random_player,
    num_games=30,
)
print("Mctx vs. random:       ", mctx_results)
print("Rollout MCTS vs. random:", rollout_results)

# Part 6: Measure batched throughput

The first call for each batch shape is an untimed JIT warm-up. Each timed result is synchronized with `block_until_ready()` and reports the median of three compiled executions. A single tic-tac-toe search may not provide enough work to offset GPU launch overhead; larger batches demonstrate the workload Mctx is designed for.

In [ ]:
benchmark_results = []
for batch_size in [1, 32, 256]:
    states = [demo_state] * batch_size
    result = rimtsaazmmtttu.benchmark_mctx_search(
        states, num_simulations=128, num_repeats=3
    )
    benchmark_results.append(result)
    print(
        f"batch={batch_size:>3}  "
        f"median={result['median_seconds']:.4f}s  "
        f"throughput={result['positions_per_second']:.1f} positions/s"
    )

# Part 7: Next steps

This tutorial supplies exact game dynamics but deliberately simple uniform priors and zero nonterminal values. The next AlphaZero milestone replaces those two placeholders with a learned policy/value network, generates self-play data from `action_weights`, and trains the network to improve future searches. The Mctx root and recurrent-function interface remains the same.